# Signature Function Tests

This notebook verifies the correctness and performance of the optimized signature function implementation.

In [1]:
import path
import ipytest
import pytest
from gaknot_lib.gaknot import GeneralizedAlgebraicKnot
from gaknot_lib.signature import SignatureFunction
from gaknot_lib.LT_signature import LT_signature_torus_knot, LT_signature_iterated_torus_knot

ipytest.autoconfig()

## Correctness Tests

In [2]:
%%ipytest -vv -W ignore::DeprecationWarning

def test_torus_knot_signature_values():
    # T(2,3) is the trefoil knot. 
    # Jumps at 1/6 and 5/6 with value -1 (h value)
    # Sigma(t) = 2 * sum(jumps before t) + jump at t
    sig = LT_signature_torus_knot(2, 3)
    
    # Before 1/6: 0
    assert sig(0.1) == 0
    # At 1/6: -1
    assert sig(1/6) == -1
    # Between 1/6 and 5/6: 2 * (-1) = -2
    assert sig(0.5) == -2
    # At 5/6: 2*(-1) + (-1) = -3
    assert sig(5/6) == -3
    # After 5/6: 2*(-1) + 2*(-1) = -4
    assert sig(0.9) == -4

def test_connected_sum_signature():
    # T(2,3) # -T(2,3) should be zero everywhere
    K = GeneralizedAlgebraicKnot([(1, [(2,3)]), (-1, [(2,3)])])
    sig = K.signature()
    assert sig.is_zero_everywhere()

def test_iterated_torus_knot_reparametrization():
    # T(2,3; 2,5) signature should match formula
    # sigma(t) = sigma_T(2,3)(2t) + sigma_T(2,5)(t)
    desc = [(2,3), (2,5)]
    sig_iterated = LT_signature_iterated_torus_knot(desc)
    
    sig_23 = LT_signature_torus_knot(2,3)
    sig_25 = LT_signature_torus_knot(2,5)
    
    # Test at some points
    for t in [0.1, 0.2, 0.3, 0.4, 0.5]:
        expected = sig_23(2*t) + sig_25(t)
        assert sig_iterated(t) == expected, f"Failed at t={t}"


======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/test
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.9
collecting ... collected 3 items

t_0880da55ef1e4287a3994b6ad2688543.py::test_torus_knot_signature_values FAILED               [ 33%]
t_0880da55ef1e4287a3994b6ad2688543.py::test_connected_sum_signature PASSED                   [ 66%]
t_0880da55ef1e4287a3994b6ad2688543.py::test_iterated_torus_knot_reparametrization PASSED     [100%]

============================================= FAILURES =============================================
_________________________________ test_torus_knot_signature_values _________________________________

    def test_torus_knot_signature_value

## Performance Tests

In [3]:
def test_large_knot_performance():
    import time
    # A knot with many jumps
    # T(91, 874) has ~80000 jumps
    start = time.time()
    sig = LT_signature_torus_knot(91, 874)
    end = time.time()
    print(f"Time to compute T(91, 874): {end - start:.4f}s")
    assert end - start < 1.0 # Should be very fast now
    
    # Evaluation performance
    start = time.time()
    for i in range(1000):
        _ = sig(i/1000)
    end = time.time()
    print(f"Time for 1000 evaluations: {end - start:.4f}s")
    assert end - start < 0.1 # Should be O(log N) now